## GPT annotation

This notebook is used to automatically annotate physical locations with GPT through OpenAI's API. The results will be used in Kertu Saul's phd thesis to automatically detect argument structures of verbs. The code was originally created by Eleri Aedmaa, minor alternations were made by Kertu Saul.

**Model**: gpt-4o

**Input**: a csv file. Delimiter is a semicolon.
* Column 1: sentence as a string.
* Column 2: word in that sentence to be annotated. Word is in the form it appeared in the sentence, aka not as a lemma. Word is a string.

**Output**: a csv file. Delimiter is a semicolon. NB! this currently causes issues as semicolons can also appear inside a sentence
* Column 1: sentence as a string.
* Column 2: word in that sentence to be annotated. Word is a string.
* Column 3: classification. Physical locations are annotated as LOC, everything else is annotated as NONE.

**NB!** Currently the output isn't entirely clean. Sometimes tags appear with quotation marks (i.e LOC", NONE"). Sometimes the sentence and word is deleted and only the tag remains. Check your output file carefully!

In [1]:
import csv
import openai
from openai import OpenAI
import chardet
import os
from dotenv import load_dotenv

In [3]:
# set up OpenAI API key to be able to use the API
load_dotenv()  # Load environment variables from .env
api_key_str = os.getenv("OPENAI_API_KEY") #get key from environment file
client = openai.OpenAI(api_key=api_key_str) #authorize myself as a client

In [5]:
def classify_location(row):
    """
    Kasutab OpenAI GPT-4 mudelit, et klassifitseerida, kas sõna on lauses füüsiline koht.
    """
    prompt = f"""Ma olen lingvist. Määra, kas järgmises lauses '{row['lause']}' olev sõna "{row['form']}" on füüsiline koht vastavalt järgmistele kategooriatele:

    Füüsilised kohad:
    1. Kohanimed (nt Bristol, Sepphoris)
    2. Ehitised/äride füüsilised asukohad (pangamaja, multimeediastuudio, Kuku klubi, arvutifirma)
    3. Füüsilised objektid, kaasa arvatud elusolendid (esikohapoodium, Kuu, varundusseade, sadul, pilv)
    4. Alad, mille geograafiline asukoht on defineeritav (põlengupaik, põhjapoolus, kaldapealne, tolmupilv)

    Mittefüüsilised kohad:
    1. Abstraktsed kohad, mille geograafilist asukohta pole võimalik määrata (nt Wifi, arvutiturg, õhuruum, digitaalplatvorm).
    2. Tegevused ja sündmused (kleidiproov, värbamine).
    3. Elusolend, kes on tegevuse tegija (rüselejal käisid sussid).
    4. Seisund (jooksevad jalad rakku, istun sitas).
    5. Viisimäärus (teravaimalt, käsikäes).
    6. Põhjuslikud määrused (hävitamisel, protsessori olemasolul).
    7. Ajamäärused (aasta, hommik).
    8. Konstruktsioonid ja stampväljendid (vaatamata hoiakule, käib jutt kehtivusest).

    Sõna: {row['form']}
    Lauses: {row['lause']}

    Vasta kujul:
    - Kui sõna on füüsiline koht: "{row['form']}|{row['lause']}|LOC"
    - Kui sõna ei ole füüsiline koht: "{row['form']}|{row['lause']}|NONE"
    NB! Vasta ALATI AINULT kujul LOC või NONE 
    """
    
    response = client.chat.completions.create(
        model="gpt-4o", #vaata palju maksab, kas mõni mudel allpool saavutab sama tulemuse
        messages=[
            {"role": "system", "content": "Sa oled lingvist, kes aitab tuvastada füüsilisi kohti."},
            {"role": "user", "content": prompt}
        ]
    )

    # Extract and return the model's response content
    return response.choices[0].message.content


In [9]:
# Sisendfail ja väljundfail
input_file = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\physical_locations_by_word\\500_sone_gptle.csv"
output_file = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\physical_locations_by_word\\500_sone_gpt_valjund.csv"

# Andmete töötlemine
with open(input_file, mode='r', encoding='utf-8-sig') as infile, open(output_file, mode='w', encoding='utf-8', newline='') as outfile:
    reader = csv.DictReader(infile, delimiter='|')
    fieldnames = ['form', 'lause', 'classification']
    writer = csv.writer(outfile, delimiter='|')
    writer.writerow(fieldnames)

    for row in reader:
        result = classify_location(row)
        writer.writerow(result.split('|'))

print(f"Töötlemine on lõpule viidud. Väljundfail salvestatud: {output_file}")

Töötlemine on lõpule viidud. Väljundfail salvestatud: C:\Users\kertu.saul\OneDrive - Eesti Keele Instituut\Dokumendid\doktoritoo\estnltk_syntax_repo_kloon\physical_location_labelling\physical_locations_by_word\500_sone_gpt_valjund.csv
